In [2]:
"""
Train random forest model using spatial splits and sklearn pipeline.
Produces:
    - cross-validation metrics
    - out-of-fold predictions
    - final test set evaluation
"""

"""
=================================
==  Primary reporting metrics  ==
=================================
OOF AUCPR — primary ranking metric

OOF Brier score — calibration/probability accuracy
OOF ECE — calibration summary
calibration plot — visual calibration check

OOF log loss — overall probabilistic performance
"""
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt
import sys
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    recall_score,
    f1_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve
from scipy.stats import randint

# define project root 
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

# Imports from project
# Use data loading helpers
from src.data.data_loading import load_full_training_pool
from src.data.data_loading import load_test_data
from src.data.data_loading import get_cv_splits

# Use feature handling helpers
from src.data.feature_handling import get_feature_columns

from config.paths import MODEL_RUNS_DIR, FIGURES_DIR, RESULTS_DIR
from config.feature_groups import TARGET_COLUMN, RADON_HIGH_CONCENTRATION, RANDOM_STATE
from config.cv_params import (
    OUTER_FOLD_COLUMN,
    INNER_CV_GROUP_COLUMN,
    INNER_CV_STRATIFY_COLUMN,
    TEST_FLAG_COLUMN,
    N_INNER_SPLITS,
)




In [ ]:
# ------------------------------------------------------------
# Run identity
# ------------------------------------------------------------
MODEL_NAME = "random_forest"   # change to: "random_forest" or "xgboost"

# nickname for run
BATCH_TAG = "overnight_2026_03_20"

SEARCH_SCORING = "neg_log_loss"
N_INNER_SPLITS = 3
N_RANDOM_SEARCH_ITERATIONS = 100


SAFE_SCORING_NAME = str(SEARCH_SCORING).replace("/", "_").replace(" ", "_")
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID = f"{MODEL_NAME}_{BATCH_TAG}_{SAFE_SCORING_NAME}_{RUN_TIMESTAMP}"

RUN_DIR = MODEL_RUNS_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_NAME:", MODEL_NAME)
print("RUN_ID:", RUN_ID)
print("RUN_DIR:", RUN_DIR)

In [ ]:
# Load the data 

# Assume that the split data is already created
# Use helper functions to load the data

train_df = load_full_training_pool().reset_index(drop=True)
# test_df  = load_test_data()

In [ ]:
# HELPER FUNCTIONS 

# Build INNER SPLITS
# These inner splits are:
#   - stratified by province
#   - grouped by spatial_cluster
#
# IMPORTANT:
# The returned indices refer to the rows of X_subset passed here.

def make_inner_cv_splits(
    X_subset: pd.DataFrame,
    stratification_column: pd.Series,
    group_column: pd.Series,
    n_splits: int = N_INNER_SPLITS,
    random_state: int = RANDOM_STATE,
):
    inner_splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    inner_splits = list(
        inner_splitter.split(
            X=X_subset,
            y=stratification_column,
            groups=group_column,
        )
    )

    return inner_splits

# Compute calibration metric
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Compute Expected Calibration Error (ECE) for binary probabilities.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bin_edges[1:-1], right=True)

    ece = 0.0

    for bin_index in range(n_bins):
        in_bin = bin_ids == bin_index
        bin_count = np.sum(in_bin)

        if bin_count > 0:
            bin_accuracy = np.mean(y_true[in_bin])
            bin_confidence = np.mean(y_prob[in_bin])
            ece += (bin_count / len(y_true)) * abs(bin_accuracy - bin_confidence)

    return ece

In [ ]:
# ------------------------------------------------------------
# Define Features and Target
# ------------------------------------------------------------

# Build feature list from config file + helper functions
# *** Do this in the pipeline preprocessor !!! ***
feature_columns = get_feature_columns(
    train_df,
    optional_excludes=None,   # standardized optional exclusion set
    extra_drop=None  # experiment-specific drops
)

# Create target
y_train_full = (train_df[TARGET_COLUMN] > RADON_HIGH_CONCENTRATION).astype(int)


# skip test data for now...
#X_test = test_df[feature_columns]
#y_test = test_df[target_column]

In [ ]:
# ------------------------------------------------------------
# Use a helper function to get split data indices that work with sklearn's cross_validate()
cv_split_idx = get_cv_splits(train_df)
# then, cross_validate(pipe, X, y, cv=cv_split_idx)


In [ ]:
# ------------------------------------------------------------
# Build Pipeline
# ------------------------------------------------------------
# For random forests:
#   - scaling is not needed
#   - median imputation is still helpful because sklearn trees do not accept NaNs

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, feature_columns),
    ],
    remainder="drop",
)

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)


In [ ]:
# ============================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================

hyperparam_distributions = {
    "classifier__n_estimators": randint(200, 1001),
    "classifier__max_depth": [None, 5, 10, 15, 20, 30],
    "classifier__min_samples_split": randint(2, 21),
    "classifier__min_samples_leaf": randint(1, 11),
    "classifier__max_features": ["sqrt", "log2", None, 0.3, 0.5, 0.8],
    "classifier__class_weight": [None, "balanced", "balanced_subsample"],
    "classifier__bootstrap": [True],
}

In [ ]:
print("train_df shape:", train_df.shape)
print("TARGET_COLUMN:", TARGET_COLUMN)
print("OUTER_FOLD_COLUMN:", OUTER_FOLD_COLUMN)

print("\nColumns in train_df:")
print(train_df.columns.tolist())

print("\ncv_fold counts:")
print(train_df[OUTER_FOLD_COLUMN].value_counts(dropna=False).sort_index())

outer_folds = sorted(train_df[OUTER_FOLD_COLUMN].dropna().unique())
print("\nouter_folds:", outer_folds)
print("number of outer folds:", len(outer_folds))

In [ ]:

# ============================================================
# NESTED CV FOLD
#    Added:
#   - fold-level metrics
#   - OOF probabilities
#   - ROC / PR / calibration artifacts
# ------------------------------------------------------------

CLASSIFICATION_THRESHOLD = 0.1
N_CALIBRATION_BINS = 10

oof_prob = np.full(len(train_df), np.nan)

outer_fold_results = []
roc_curves = []
pr_curves = []

outer_folds = sorted(train_df[OUTER_FOLD_COLUMN].dropna().unique())

print("About to start nested CV")
print("outer_folds:", outer_folds)
print("number of train rows:", len(train_df))

for outer_fold in outer_folds:

    print("\n" + "=" * 60)
    print(f"OUTER FOLD {outer_fold}")
    print("=" * 60)

    train_mask = train_df[OUTER_FOLD_COLUMN] != outer_fold
    val_mask = train_df[OUTER_FOLD_COLUMN] == outer_fold

    # Outer train
    X_tr = train_df.loc[train_mask, feature_columns].reset_index(drop=True)
    y_tr = y_train_full.loc[train_mask].reset_index(drop=True)

    # Outer validation
    X_val = train_df.loc[val_mask, feature_columns]
    y_val = y_train_full.loc[val_mask]

    # Inner fold metadata
    stratify_tr = train_df.loc[train_mask, INNER_CV_STRATIFY_COLUMN].reset_index(drop=True)
    groups_tr = train_df.loc[train_mask, INNER_CV_GROUP_COLUMN].reset_index(drop=True)

    inner_cv_splits = make_inner_cv_splits(
        X_subset=X_tr,
        stratification_column=stratify_tr,
        group_column=groups_tr,
        n_splits=N_INNER_SPLITS,
        random_state=RANDOM_STATE,
    )

    print("Outer-train rows:", len(X_tr))
    print("Outer-validation rows:", len(X_val))
    print("Number of inner splits:", len(inner_cv_splits))
    print("Outer-train positive rate:", y_tr.mean())
    print("Outer-validation positive rate:", y_val.mean())

    random_search = RandomizedSearchCV(
        estimator=random_forest_pipeline,
        param_distributions=hyperparam_distributions,
        n_iter=N_RANDOM_SEARCH_ITERATIONS,
        cv=inner_cv_splits,
        scoring=SEARCH_SCORING,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
        verbose=1,
    )

    random_search.fit(X_tr, y_tr)

    best_model = random_search.best_estimator_
    best_params = random_search.best_params_
    best_inner_score = random_search.best_score_

    print("Best params:", best_params)
    print("Best inner mean search score:", best_inner_score)

    y_val_prob = best_model.predict_proba(X_val)[:, 1]
    y_val_pred = (y_val_prob >= CLASSIFICATION_THRESHOLD).astype(int)

    oof_prob[np.where(val_mask)[0]] = y_val_prob

    fold_roc_auc = roc_auc_score(y_val, y_val_prob)
    fold_aucpr = average_precision_score(y_val, y_val_prob)
    fold_logloss = log_loss(y_val, y_val_prob)
    fold_brier = brier_score_loss(y_val, y_val_prob)
    fold_ece = expected_calibration_error(y_val, y_val_prob, n_bins=N_CALIBRATION_BINS)
    fold_accuracy = accuracy_score(y_val, y_val_pred)
    fold_recall = recall_score(y_val, y_val_pred, zero_division=0)
    fold_f1 = f1_score(y_val, y_val_pred, zero_division=0)

    fold_fpr, fold_tpr, _ = roc_curve(y_val, y_val_prob)
    roc_curves.append((fold_fpr, fold_tpr))

    fold_precision, fold_recall_curve, _ = precision_recall_curve(y_val, y_val_prob)
    pr_curves.append((fold_recall_curve, fold_precision))

    fold_result = {
        "model_name": MODEL_NAME,
        "run_id": RUN_ID,
        "outer_fold": int(outer_fold),
        "n_train": int(len(X_tr)),
        "n_val": int(len(X_val)),
        "train_positive_rate": float(y_tr.mean()),
        "val_positive_rate": float(y_val.mean()),
        "roc_auc": float(fold_roc_auc),
        "aucpr": float(fold_aucpr),
        "log_loss": float(fold_logloss),
        "brier": float(fold_brier),
        "ece": float(fold_ece),
        "accuracy": float(fold_accuracy),
        "recall": float(fold_recall),
        "f1": float(fold_f1),
        "best_params_json": json.dumps(best_params, sort_keys=True, default=str),
    }

    outer_fold_results.append(fold_result)

    print(
        f"Fold {outer_fold} | "
        f"ROC-AUC: {fold_roc_auc:.4f} | "
        f"AUCPR: {fold_aucpr:.4f} | "
        f"LogLoss: {fold_logloss:.4f} | "
        f"Brier: {fold_brier:.4f} | "
        f"ECE: {fold_ece:.4f} | "
        f"Accuracy: {fold_accuracy:.4f} | "
        f"Recall: {fold_recall:.4f} | "
        f"F1: {fold_f1:.4f}",
        flush=True,
    )

    outer_fold_results_df = pd.DataFrame(outer_fold_results)
    outer_fold_results_df.to_csv(RUN_DIR / "outer_fold_results.csv", index=False)

print("\n" + "=" * 60)
print("FOLD-LEVEL RESULTS")
print("=" * 60)
display(outer_fold_results_df)


In [ ]:
oof_predictions_df = pd.DataFrame({
    "model_name": MODEL_NAME,
    "run_id": RUN_ID,
    "row_index": train_df.index,
    "cv_fold": train_df[OUTER_FOLD_COLUMN].values,
    "y_true": y_train_full.values,
    "oof_prob": oof_prob,
    "oof_pred_at_threshold": (oof_prob >= CLASSIFICATION_THRESHOLD).astype(int),
})

for col in ["FSA", "province", TARGET_COLUMN]:
    if col in train_df.columns:
        oof_predictions_df[col] = train_df[col].values

oof_predictions_df.to_csv(RUN_DIR / "oof_predictions.csv", index=False)


In [ ]:
print("Search scoring metric:", random_search.scoring)
print("Classification threshold:", CLASSIFICATION_THRESHOLD)
print("Max predicted probability in last outer fold:", y_val_prob.max())
print(f"Number >= {CLASSIFICATION_THRESHOLD}:", np.sum(y_val_prob >= CLASSIFICATION_THRESHOLD))
print("Quantiles:", np.quantile(y_val_prob, [0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]))


In [ ]:
# ------------------------------------------------------------
# Overall OOF metrics
# ------------------------------------------------------------

if np.any(np.isnan(oof_prob)):
    raise ValueError("Some OOF probabilities are still NaN.")

train_df = train_df.copy()
train_df["oof_prob"] = oof_prob
train_df["oof_pred_at_threshold"] = (train_df["oof_prob"] >= CLASSIFICATION_THRESHOLD).astype(int)

oof_roc_auc = roc_auc_score(y_train_full, train_df["oof_prob"])
oof_aucpr = average_precision_score(y_train_full, train_df["oof_prob"])
oof_logloss = log_loss(y_train_full, train_df["oof_prob"])
oof_brier = brier_score_loss(y_train_full, train_df["oof_prob"])
oof_ece = expected_calibration_error(y_train_full, train_df["oof_prob"], n_bins=N_CALIBRATION_BINS)

oof_accuracy = accuracy_score(y_train_full, train_df["oof_pred_at_threshold"])
oof_recall = recall_score(y_train_full, train_df["oof_pred_at_threshold"], zero_division=0)
oof_f1 = f1_score(y_train_full, train_df["oof_pred_at_threshold"], zero_division=0)

print("\n" + "=" * 60)
print("OVERALL OOF METRICS")
print("=" * 60)
print("Positive prevalence:", y_train_full.mean())
print("OOF ROC-AUC:", oof_roc_auc)
print("OOF AUCPR:", oof_aucpr)
print("OOF LogLoss:", oof_logloss)
print("OOF Brier:", oof_brier)
print("OOF ECE:", oof_ece)
print("OOF Accuracy:", oof_accuracy)
print("OOF Recall:", oof_recall)
print("OOF F1:", oof_f1)

In [ ]:
thresholds_to_check = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

for threshold in thresholds_to_check:
    oof_pred = (train_df["oof_prob"] >= threshold).astype(int)

    threshold_accuracy = accuracy_score(y_train_full, oof_pred)
    threshold_recall = recall_score(y_train_full, oof_pred, zero_division=0)
    threshold_f1 = f1_score(y_train_full, oof_pred, zero_division=0)

    print(
        f"Threshold {threshold:.2f} | "
        f"Accuracy: {threshold_accuracy:.4f} | "
        f"Recall: {threshold_recall:.4f} | "
        f"F1: {threshold_f1:.4f} | "
        f"Predicted positives: {oof_pred.sum()}"
    )

In [ ]:
# ------------------------------------------------------------
# Mean ± std across outer folds
# ------------------------------------------------------------

metric_columns = [
    "roc_auc",
    "aucpr",
    "log_loss",
    "brier",
    "ece",
    "accuracy",
    "recall",
    "f1",
]

print("\n" + "=" * 60)
print("MEAN PERFORMANCE ACROSS OUTER FOLDS")
print("=" * 60)

for metric_name in metric_columns:
    metric_mean = outer_fold_results_df[metric_name].mean()
    metric_std = outer_fold_results_df[metric_name].std()
    print(f"{metric_name}: {metric_mean:.4f} ± {metric_std:.4f}")


In [ ]:
# ------------------------------------------------------------
# ROC curves for all outer folds
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

for i, (fpr, tpr) in enumerate(roc_curves, start=1):
    plt.plot(fpr, tpr, label=f"Fold {i}")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Nested CV)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# ------------------------------------------------------------
# Precision-Recall curves for all outer folds
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

for i, (recall_curve_values, precision_values) in enumerate(pr_curves, start=1):
    plt.plot(recall_curve_values, precision_values, label=f"Fold {i}")

baseline_prevalence = y_train_full.mean()
plt.axhline(y=baseline_prevalence, linestyle="--", linewidth=1, label="Prevalence baseline")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves (Nested CV)")
plt.legend(loc="best")
plt.grid(True)
plt.show()

In [ ]:
# ------------------------------------------------------------
# Calibration plot from OOF probabilities
# ------------------------------------------------------------

fraction_of_positives, mean_predicted_value = calibration_curve(
    y_train_full,
    train_df["oof_prob"],
    n_bins=N_CALIBRATION_BINS,
    strategy="uniform",
)

plt.figure(figsize=(7, 7))
plt.plot(mean_predicted_value, fraction_of_positives, marker="o", label="OOF calibration")
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Calibration Plot (OOF probabilities)")
plt.legend(loc="best")
plt.grid(True)
plt.show()

In [ ]:
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_train_full,
    train_df["oof_prob"],
    n_bins=10,
    strategy="quantile",
)

plt.figure(figsize=(7, 7))
plt.plot(mean_predicted_value, fraction_of_positives, marker="o", label="OOF calibration")
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")

plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Calibration Plot (OOF probabilities)")
plt.legend(loc="best")
plt.grid(True)
plt.show()

In [ ]:
summary_results = {
    "model_name": MODEL_NAME,
    "run_id": RUN_ID,
    "batch_tag": BATCH_TAG,
    "search_scoring": SEARCH_SCORING,
    "n_rows": int(len(train_df)),
    "positive_rate": float(y_train_full.mean()),
    "n_outer_folds": int(train_df[OUTER_FOLD_COLUMN].nunique()),
    "classification_threshold": float(CLASSIFICATION_THRESHOLD),
    "n_calibration_bins": int(N_CALIBRATION_BINS),
    "oof_roc_auc": float(oof_roc_auc),
    "oof_aucpr": float(oof_aucpr),
    "oof_log_loss": float(oof_logloss),
    "oof_brier": float(oof_brier),
    "oof_ece": float(oof_ece),
    "oof_accuracy": float(oof_accuracy),
    "oof_recall": float(oof_recall),
    "oof_f1": float(oof_f1),
    "mean_fold_roc_auc": float(pd.DataFrame(outer_fold_results)["roc_auc"].mean()),
    "mean_fold_aucpr": float(pd.DataFrame(outer_fold_results)["aucpr"].mean()),
    "mean_fold_log_loss": float(pd.DataFrame(outer_fold_results)["log_loss"].mean()),
    "mean_fold_brier": float(pd.DataFrame(outer_fold_results)["brier"].mean()),
    "mean_fold_ece": float(pd.DataFrame(outer_fold_results)["ece"].mean()),
}

with open(RUN_DIR / "summary_metrics.json", "w") as f:
    json.dump(summary_results, f, indent=2)

pd.DataFrame([summary_results]).to_csv(RUN_DIR / "summary_metrics.csv", index=False)


In [ ]:
# ------------------------------------------------------------
# Optional: inspect feature importances from a final refit on all training data
# ------------------------------------------------------------
# This is NOT an unbiased performance estimate.
# It is just a model-interpretation step after your OOF evaluation.

final_random_forest_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=hyperparam_distributions,
    n_iter=N_RANDOM_SEARCH_ITERATIONS,
    cv=make_inner_cv_splits(
        X_subset=train_df[feature_columns].reset_index(drop=True),
        stratification_column=train_df[INNER_CV_STRATIFY_COLUMN].reset_index(drop=True),
        group_column=train_df[INNER_CV_GROUP_COLUMN].reset_index(drop=True),
        n_splits=N_INNER_SPLITS,
        random_state=RANDOM_STATE,
    ),
    scoring=SEARCH_SCORING,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

final_random_forest_search.fit(train_df[feature_columns], y_train_full)

final_random_forest_model = final_random_forest_search.best_estimator_
final_random_forest_classifier = final_random_forest_model.named_steps["classifier"]

feature_importance_df = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance": final_random_forest_classifier.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(feature_importance_df.head(20))


In [ ]:
# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

train_df.to_csv(RUN_DIR / "randomforest_oof_predictions_with_metadata.csv", index=False)
outer_fold_results_df.to_csv(RUN_DIR / "randomforest_outer_fold_results.csv", index=False)
oof_predictions_df.to_csv(RUN_DIR / "randomforest_oof_predictions.csv", index=False)

summary_results_randomforest = {
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "search_scoring": SEARCH_SCORING,
    "n_random_search_iterations": N_RANDOM_SEARCH_ITERATIONS,
    "n_inner_splits": N_INNER_SPLITS,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "n_calibration_bins": N_CALIBRATION_BINS,
    "oof_roc_auc": oof_roc_auc,
    "oof_aucpr": oof_aucpr,
    "oof_logloss": oof_logloss,
    "oof_brier": oof_brier,
    "oof_ece": oof_ece,
    "oof_accuracy": oof_accuracy,
    "oof_recall": oof_recall,
    "oof_f1": oof_f1,
    "prevalence": y_train_full.mean(),
    "n_train_rows": len(train_df),
}

summary_results_randomforest_df = pd.DataFrame([summary_results_randomforest])
summary_results_randomforest_df.to_csv(RUN_DIR / "randomforest_summary_results.csv", index=False)

if "feature_importance_df" in globals():
    feature_importance_df.to_csv(RUN_DIR / "randomforest_feature_importance.csv", index=False)

display(summary_results_randomforest_df)
